# Path Planning On a Grid :: Jump Point Search


<center><img src="./img/1.png"/></center>


In this lab, we extend the A* implementation from lab03 with **Jump Point Search (JPS)**, following [Harabor and Grastien (2011)](https://doi.org/10.1609/aaai.v25i1.7994).

We use an **8-connected grid**: cardinal moves cost 1, diagonal moves cost $\sqrt{2}$, and diagonal corner crossing is allowed. JPS skips symmetric routes by scanning straight and diagonal directions, adding only jump points to OPEN. The resulting path remains optimal and is expanded into individual cells for drawing and validation.

Complete the marked code fragments, reuse your A* work from lab03, and compare both algorithms on the supplied tasks.


In [ ]:
from heapq import heappop, heappush
from typing import Callable, Dict, Iterable, List, Optional, Tuple, Type, Union

import matplotlib.pyplot as plt
import numpy as np
import numpy.typing as npt

%matplotlib inline

## Grid Map Representation

In [ ]:
from jps_utils.map import Map
from jps_utils.utils import convert_string_to_cells

In [ ]:
help(Map)

In [ ]:
from jps_utils.utils import compute_cost

Moves are cardinal or diagonal. Cardinal moves cost 1, diagonal moves cost $\sqrt{2}$, and `compute_cost` also evaluates the longer straight or diagonal segments returned by JPS.


In [ ]:
help(compute_cost)

## Search Node

Reuse the node from lab03. Its ordering compares f-values.


In [ ]:
from jps_utils.node import BaseNode

class Node(BaseNode):
    def __lt__(self, other):
        """
        Compares the keys of two nodes, which is needed for sorting/extracting the best element from OPEN.
        """
        # <YOUR CODE GOES HERE>
        pass

Nodes can be used as dictionary keys through the inherited hash and equality methods.


In [ ]:
help(Node.__hash__)

## Visualization

In [ ]:
from jps_utils.utils import draw, make_path


Example of the visualization

In [ ]:
height = 15
width = 30
map_str = """
. . . . . . . . . . . . . . . . . . . . . # # . . . . . . .  
. . . . . . . . . . . . . . . . . . . . . # # . . . . . . . 
. . . . . . . . . . . . . . . . . . . . . # # . . . . . . . 
. . . # # . . . . . . . . . . . . . . . . # # . . . . . . . 
. . . # # . . . . . . . . # # . . . . . . # # . . . . . . . 
. . . # # . . . . . . . . # # . . . . . . # # # # # . . . . 
. . . # # . . . . . . . . # # . . . . . . # # # # # . . . . 
. . . # # . . . . . . . . # # . . . . . . . . . . . . . . . 
. . . # # . . . . . . . . # # . . . . . . . . . . . . . . . 
. . . # # . . . . . . . . # # . . . . . . . . . . . . . . . 
. . . # # . . . . . . . . # # . . . . . . . . . . . . . . . 
. . . # # . . . . . . . . # # . . . . . . . . . . . . . . . 
. . . . . . . . . . . . . # # . . . . . . . . . . . . . . . 
. . . . . . . . . . . . . # # . . . . . . . . . . . . . . .
. . . . . . . . . . . . . # # . . . . . . . . . . . . . . .
"""
cells = convert_string_to_cells(map_str)

test_map = Map(cells)
start = Node(1, 1)
goal = Node(13, 28)
draw(test_map, start, goal)

## Search Tree (OPEN and CLOSED)

Reuse your priority-queue search tree from lab03.


In [ ]:
class SearchTreePQD:
    """
    SearchTree using a priority queue for OPEN and a dictionary for CLOSED.
    """

    def __init__(self):
        self._open = []  # Priority queue for nodes in OPEN
        self._closed = {}  # Dictionary for nodes in CLOSED (expanded nodes)
        self._enc_open_dublicates = 0  # Number of dublicates encountered in OPEN

    def __len__(self) -> int:
        """
        Returns the size of the search tree. Useful for assessing the memory
        footprint of the algorithm, especially at the final iteration.
        """
        return len(self._open) + len(self._closed)

    def open_is_empty(self) -> bool:
        """
        Checks if OPEN is empty.
        If true, the main search loop should be interrupted.
        """
        # <YOUR CODE GOES HERE>
        pass

    def add_to_open(self, item: Node):
        """
        Adds a node to the search tree, specifically to OPEN. This node is either
        entirely new or a duplicate of an existing node in OPEN.
        This implementation detects duplicates lazily; thus, nodes are added to
        OPEN without initial duplicate checks.
        """
        # <YOUR CODE GOES HERE>
        pass

    def get_best_node_from_open(self) -> Optional[Node]:
        """
        Retrieves the best node from OPEN, defined by the minimum key.
        This node will then be expanded in the main search loop.

        Duplicates are managed here. If a node has been expanded previously
        (and is in CLOSED), it's skipped and the next best node is considered.

        Returns None if OPEN is empty.
        """
        # <YOUR CODE GOES HERE>
        pass


    def add_to_closed(self, item: Node):
        """
        Adds a node to the CLOSED dictionary.
        """
        # <YOUR CODE GOES HERE>
        pass

    def was_expanded(self, item: Node) -> bool:
        """
        Checks if a node has been previously expanded.
        """
        # <YOUR CODE GOES HERE>
        pass

    @property
    def opened(self):
        return self._open

    @property
    def expanded(self):
        return self._closed.values()

    @property
    def number_of_open_dublicates(self):
        return self._enc_open_dublicates

## Validating the Results

The lab03 test helpers use the same six-value search result as A* and JPS. The JPS review checks path validity, optimality, and pruning.


In [ ]:
from tests.lab_test import simple_test, simple_test_not_found, massive_test
from review_checks import compare_on_tasks, review_jps


## A* Baseline


For 8-connected movement, use the **octile distance** as the heuristic. If $d_i=|i_1-i_2|$ and $d_j=|j_1-j_2|$, the obstacle-free path uses $\min(d_i,d_j)$ diagonal moves and $|d_i-d_j|$ cardinal moves. Thus $h=\sqrt{2}\min(d_i,d_j)+|d_i-d_j|$. This is admissible and consistent for the grid used here.


In [ ]:
def octile_distance(i1: int, j1: int, i2: int, j2: int) -> float:
    """Compute the octile distance between two cells.

    Parameters
    ----------
    i1, j1 : int, int
        Coordinates of the first cell.
    i2, j2 : int, int
        Coordinates of the second cell.

    Returns
    -------
    float
        Exact obstacle-free distance for cardinal cost 1 and diagonal cost sqrt(2).
    """
    # <YOUR CODE GOES HERE>
    pass


Adapt your lab03 A* implementation to the 8-connected grid. Keep its six-value return interface so the same checks can compare it with JPS.


In [ ]:
def astar(
    task_map: Map,
    start_i: int,
    start_j: int,
    goal_i: int,
    goal_j: int,
    heuristic_func: Callable,
    search_tree: Type[SearchTreePQD],
) -> Tuple[bool, Optional[Node], int, int, Optional[Iterable[Node]], Optional[Iterable[Node]]]:
    """
    Implements the A* search algorithm.

    Parameters
    ----------
    task_map : Map
        The grid or map being searched.
    start_i, start_j : int, int
        Starting coordinates.
    goal_i, goal_j : int, int
        Goal coordinates.
    heuristic_func : Callable
        Heuristic function for estimating the distance from a node to the goal.
    search_tree : Type[SearchTreePQD]
        The search tree to use.

    Returns
    -------
    Tuple[bool, Optional[Node], int, int, Optional[Iterable[Node]], Optional[Iterable[Node]]]
        Tuple containing:
        - A boolean indicating if a path was found.
        - The last node in the found path or None.
        - Number of algorithm iterations.
        - Size of the resultant search tree.
        - OPEN set nodes for visualization or None.
        - CLOSED set nodes.
    """
    # <YOUR CODE GOES HERE>
    pass


## Jump Point Search

JPS8 replaces ordinary neighbor expansion with natural and forced directions. Scans stop at the goal or the next forced jump point.


JPS uses the same arguments and six-value return interface as the A* baseline. Implement the jump scan, successor pruning, and search loop below.


In [ ]:
def _sign(value: int) -> int:
    """Return -1, 0, or 1 according to the sign of value."""
    return (value > 0) - (value < 0)


def jump(
    task_map: Map,
    i: int,
    j: int,
    di: int,
    dj: int,
    goal_i: int,
    goal_j: int,
) -> Optional[Tuple[int, int]]:
    """Scan in one of eight directions and return the next jump point.

    Parameters
    ----------
    task_map : Map
        The 8-connected grid being searched.
    i, j : int, int
        Coordinates from which the scan starts.
    di, dj : int, int
        Scan direction; each component belongs to {-1, 0, 1}, and they are not both zero.
    goal_i, goal_j : int, int
        Coordinates of the goal cell.

    Returns
    -------
    Optional[Tuple[int, int]]
        Coordinates of the first goal or forced jump point encountered, or None
        if the scan reaches an obstacle or the map boundary first.
    """
    # <YOUR CODE GOES HERE>
    raise NotImplementedError



def get_successor_directions(task_map: Map, node: Node) -> List[Tuple[int, int]]:
    """Return natural and forced directions for an 8-connected JPS node.

    Parameters
    ----------
    task_map : Map
        The grid being searched.
    node : Node
        Current jump point. Its parent determines the incoming direction.

    Returns
    -------
    List[Tuple[int, int]]
        Direction vectors that must be scanned without losing an optimal path.
    """
    # <YOUR CODE GOES HERE>
    raise NotImplementedError


In [ ]:
def jps(
    task_map: Map,
    start_i: int,
    start_j: int,
    goal_i: int,
    goal_j: int,
    heuristic_func: Callable,
    search_tree: Type[SearchTreePQD],
) -> Tuple[bool, Optional[Node], int, int, Optional[Iterable[Node]], Optional[Iterable[Node]]]:
    """Implement Jump Point Search on a uniform-cost 8-connected grid.

    Parameters
    ----------
    task_map : Map
        The grid being searched.
    start_i, start_j : int, int
        Coordinates of the start cell.
    goal_i, goal_j : int, int
        Coordinates of the goal cell.
    heuristic_func : Callable
        Function estimating the remaining cost from a cell to the goal.
    search_tree : Type[SearchTreePQD]
        Search-tree class providing OPEN and CLOSED operations.

    Returns
    -------
    Tuple[bool, Optional[Node], int, int, Optional[Iterable[Node]], Optional[Iterable[Node]]]
        A tuple containing, in order:
        - whether a path was found;
        - the goal node, or None when no path exists;
        - number of main-loop iterations;
        - final search-tree size;
        - OPEN nodes for visualization, or None on failure;
        - CLOSED nodes for visualization.
    """
    # <YOUR CODE GOES HERE>
    raise NotImplementedError


## Visual Test

Use the same lab03 test and drawing functions for both algorithms.


In [ ]:
comparison_cells = np.zeros((40, 60), dtype=np.int8)

# Four walls with staggered gates create a clear ascending route. The larger
# open regions also make the difference in explored nodes easy to see.
for column, gate_center in ((12, 32), (24, 24), (36, 16), (48, 8)):
    comparison_cells[:, column] = 1
    comparison_cells[gate_center - 3 : gate_center + 4, column] = 0

comparison_map = Map(comparison_cells)
comparison_start = Node(36, 3)
comparison_goal = Node(3, 56)

for algorithm_name, search_function in (("A*", astar), ("Jump Point Search", jps)):
    result = search_function(
        comparison_map,
        comparison_start.i,
        comparison_start.j,
        comparison_goal.i,
        comparison_goal.j,
        octile_distance,
        SearchTreePQD,
    )
    found, end_node, steps, tree_size, opened, expanded = result
    assert found
    path, path_length = make_path(end_node)
    print(f"{algorithm_name}: length={path_length:.3f}, steps={steps}, tree size={tree_size}")
    draw(
        comparison_map,
        comparison_start,
        comparison_goal,
        path,
        opened,
        expanded,
    )


## Semi-automatic Review

The review checks the shared interface, path validity, optimal costs, unreachable tasks, and JPS pruning.


In [ ]:
review_jps(Map, Node, SearchTreePQD, octile_distance, astar, jps)


## Performance on Ten Tasks

Tasks 0–8 reuse the lab03 maps and endpoints; their reference costs are recomputed for 8-connected movement. Task 9 is a 256 × 256 warehouse floor with regularly spaced shelf banks, wide aisles, and alternating shelf offsets. It provides a structured performance comparison with several route choices. The comparison checks identical optimal lengths, fewer JPS expansions, and lower total JPS runtime.


In [ ]:
comparison = compare_on_tasks(
    astar,
    jps,
    "data/",
    10,
    octile_distance,
    SearchTreePQD,
)
comparison


## Questions

1. Which grid-path symmetries does JPS remove?
2. What makes a neighbor forced?
3. Why does skipping intermediate cells preserve optimality?
4. Compare A* and JPS path length, expansions, search-tree size, and runtime on the ten tasks.


## Your answers

Your answer goes here.
